# Zernike PSF Moments (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-02-24  
**Last Modified:** 2026-03-19  
**Status:** Complete  
**Keywords:** PSF, Zernike, GalSim, moments, FWHM, ellipticity, annular Zernike  

## Description

Simulate the Rubin Observatory PSF from annular Zernike wavefront coefficients
and measure its moments (FWHM, ellipticity) using GalSim and HSM.

Key functionality:
1. Draw optical+atmospheric PSF from Zernike coefficients (Z4–Z22)
2. Measure FWHM, ellipticity (e1, e2), and centroid via HSM adaptive moments
3. Interactive widget interface for exploring coefficient space

**Output:** PSF images, moment measurements  
**Based on:** Original `zernike_psf_moments.ipynb` in top-level working area

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-02-24 | Aaron Roodman | Initial version |
| 2026-03-19 | Aaron Roodman | Refactored to notebook template style, moved to psf/ |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Atmospheric Reference PSF](#atm-ref)
5. [Interactive PSF Explorer](#interactive)

<a id='params'></a>
## 1. Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# Telescope parameters (Rubin Observatory / LSSTCam)
LAM_NM      = 500.0   # wavelength [nm]
DIAM_M      = 8.36    # primary mirror diameter [m]
OBSCURATION = 0.61    # central obscuration ratio
PIXEL_SCALE = 0.2     # arcsec/pixel
STAMP_SIZE  = 128     # PSF image size [pixels]

# Atmospheric turbulence
ATM_R0      = 0.1382  # Fried parameter r0 at 500 nm [m]
ATM_L0_M    = 30.0    # outer scale L₀ [m]

# Default Zernike test values (microns)
DEFAULT_ZERNIKES = {4: 0.10, 7: 0.08}  # Z4=defocus, Z7=coma

<a id='setup'></a>
## 2. Setup & Imports

In [ ]:
import numpy as np
import galsim
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

try:
    from lsst.ts.wep.utils import convertZernikesToPsfWidth
    HAS_TS_WEP = True
    print('ts_wep available')
except ImportError:
    HAS_TS_WEP = False
    print('ts_wep not available')

LAM_MICRONS = LAM_NM / 1000.0

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

<a id='functions'></a>
## 3. Helper Functions

In [ ]:
# Zernike name mapping (Noll indices Z4–Z22)
NOLL_NAMES = {
    4:  'Defocus', 5:  'Oblique astigmatism', 6:  'Vertical astigmatism',
    7:  'Vertical coma', 8:  'Horizontal coma', 9:  'Vertical trefoil',
    10: 'Horizontal trefoil', 11: 'Primary spherical',
    12: 'Vert 2nd astig', 13: 'Oblq 2nd astig', 14: 'Oblq trefoil',
    15: 'Vert trefoil', 16: 'Oblq quadrafoil', 17: 'Vert quadrafoil',
    18: 'Oblq 2nd coma', 19: 'Vert 2nd coma', 20: 'Oblq 3rd astig',
    21: 'Vert 3rd astig', 22: 'Secondary spherical'
}


def draw_psf(aberrations_waves, nx=128, ny=128, scale=0.2,
             lam=LAM_NM, diam=DIAM_M, obscuration=OBSCURATION,
             atm_r0=ATM_R0, atm_L0=ATM_L0_M, include_atm=True):
    """Draw PSF from Zernike aberrations in waves.

    Parameters
    ----------
    aberrations_waves : array-like
        GalSim-style aberration array (index = Noll index, values in waves).
    nx, ny : int
        Stamp size in pixels.
    scale : float
        Pixel scale in arcsec/pixel.
    lam, diam, obscuration : float
        Telescope parameters.
    atm_r0, atm_L0 : float
        Atmospheric turbulence parameters.
    include_atm : bool
        If True, convolve with atmospheric VonKarman PSF.

    Returns
    -------
    galsim.Image
        Drawn PSF image.
    """
    optical_psf = galsim.OpticalPSF(
        lam=lam, diam=diam, obscuration=obscuration,
        aberrations=aberrations_waves, annular_zernike=True
    )
    if include_atm:
        atm = galsim.VonKarman(lam=lam, r0=atm_r0, L0=atm_L0)
        psf = galsim.Convolve(atm, optical_psf)
    else:
        psf = optical_psf
    return psf.drawImage(nx=nx, ny=ny, method='no_pixel', scale=scale)


def zernike_coeffs_to_aberrations(coeffs_um, lam_um=LAM_MICRONS):
    """Convert {noll_index: coeff_um} dict to GalSim aberrations array in waves.

    Parameters
    ----------
    coeffs_um : dict
        {noll_index: coefficient_in_microns}
    lam_um : float
        Wavelength in microns.

    Returns
    -------
    aberrations : ndarray
        Array of length noll_max+1, values in waves.
    """
    if not coeffs_um:
        return np.zeros(4)
    noll_max = max(coeffs_um.keys())
    aberrations = np.zeros(noll_max + 1)
    for noll_j, coeff in coeffs_um.items():
        aberrations[noll_j] = coeff / lam_um
    return aberrations


def measure_psf_moments(image, pixel_scale=0.2):
    """Measure FWHM, ellipticity, and centroid using HSM adaptive moments.

    Parameters
    ----------
    image : galsim.Image
        PSF image to measure.
    pixel_scale : float
        Pixel scale in arcsec/pixel.

    Returns
    -------
    fwhm : float
        FWHM in arcsec.
    e1, e2 : float
        Ellipticity components.
    cx, cy : float
        Centroid offset from image center in arcsec.
    """
    try:
        shape = galsim.hsm.FindAdaptiveMom(image)
        sigma_pix = shape.moments_sigma
        fwhm = 2.0 * np.sqrt(2.0 * np.log(2.0)) * sigma_pix * pixel_scale
        e1 = shape.observed_shape.e1
        e2 = shape.observed_shape.e2
        cx = (shape.moments_centroid.x - image.array.shape[1] / 2) * pixel_scale
        cy = (shape.moments_centroid.y - image.array.shape[0] / 2) * pixel_scale
        return fwhm, e1, e2, cx, cy
    except Exception:
        return np.nan, np.nan, np.nan, 0.0, 0.0


def draw_and_measure(coeffs_um, stamp_size=STAMP_SIZE, pixel_scale=PIXEL_SCALE,
                     include_atm=True):
    """Draw PSF from Zernike coefficients (microns) and measure moments.

    Parameters
    ----------
    coeffs_um : dict
        {noll_index: coefficient_in_microns}
    stamp_size : int
        Image stamp size in pixels.
    pixel_scale : float
        Pixel scale in arcsec/pixel.
    include_atm : bool
        Include atmospheric component.

    Returns
    -------
    image : galsim.Image
        Drawn PSF image.
    fwhm : float
        FWHM in arcsec.
    e1, e2 : float
        Ellipticity components.
    """
    aberrations = zernike_coeffs_to_aberrations(coeffs_um)
    image = draw_psf(aberrations, nx=stamp_size, ny=stamp_size, scale=pixel_scale,
                     include_atm=include_atm)
    fwhm, e1, e2, cx, cy = measure_psf_moments(image, pixel_scale)
    return image, fwhm, e1, e2

<a id='atm-ref'></a>
## 4. Atmospheric Reference PSF

Calculate the atmosphere-only PSF FWHM (used for quadrature subtraction).

In [ ]:
psf_atm_ref = draw_psf(np.zeros(12), nx=STAMP_SIZE, ny=STAMP_SIZE,
                        scale=PIXEL_SCALE, include_atm=True)
fwhm_atm, _, _, _, _ = measure_psf_moments(psf_atm_ref, PIXEL_SCALE)
print(f'Atmospheric FWHM: {fwhm_atm:.3f} arcsec  (r0={ATM_R0} m, L0={ATM_L0_M} m at {LAM_NM} nm)')

<a id='interactive'></a>
## 5. Interactive PSF Explorer

Enter Zernike coefficients (microns) and click **Generate PSF** to visualize.

In [ ]:
# Create interactive input widgets for Z4–Z22
zernike_inputs = {}
for noll_j in range(4, 23):
    name = NOLL_NAMES.get(noll_j, f'Z{noll_j}')
    default_val = DEFAULT_ZERNIKES.get(noll_j, 0.0)
    zernike_inputs[noll_j] = widgets.FloatText(
        value=default_val,
        description=f'Z{noll_j} ({name}):',
        style={'description_width': '200px'},
        layout=widgets.Layout(width='350px'),
        step=0.01
    )

generate_button = widgets.Button(
    description='Generate PSF', button_style='success', icon='refresh',
    layout=widgets.Layout(width='200px', height='40px')
)
output_area = widgets.Output()


def on_generate_button_clicked(b):
    with output_area:
        clear_output(wait=True)
        coeffs_um = {j: w.value for j, w in zernike_inputs.items() if w.value != 0.0}
        if not coeffs_um:
            print('No non-zero Zernike coefficients specified.')
            return

        image, fwhm_total, e1, e2 = draw_and_measure(coeffs_um)
        e_mag = np.sqrt(e1**2 + e2**2)
        fwhm_optics_quad = np.sqrt(np.abs(fwhm_total**2 - fwhm_atm**2))

        print(f'Total FWHM: {fwhm_total:.3f}"  |  Optics (quad): {fwhm_optics_quad:.3f}"'
              f'  |  e1={e1:+.4f}  e2={e2:+.4f}  |e|={e_mag:.4f}')

        fig, ax = plt.subplots(1, 1, figsize=(7, 6))
        arr = image.array
        ny, nx = arr.shape
        half = int(3.0 / PIXEL_SCALE)
        c = ny // 2
        extent = [-3, 3, -3, 3]
        im = ax.imshow(arr[c-half:c+half, c-half:c+half],
                       origin='lower', extent=extent, cmap='hot')
        ax.set_xlabel('x (arcsec)')
        ax.set_ylabel('y (arcsec)')
        ax.set_title(f'PSF  FWHM={fwhm_total:.3f}"  |e|={e_mag:.3f}')
        plt.colorbar(im, ax=ax, label='Intensity')

        # FWHM circle
        circle = Circle((0, 0), fwhm_total / 2, fill=False, edgecolor='red',
                        linewidth=1.5, linestyle='--')
        ax.add_patch(circle)

        # Ellipticity whisker
        if e_mag > 0.001:
            e_angle = 0.5 * np.arctan2(e2, e1)
            length = 1.5 * fwhm_total / 2 * (1 + e_mag)
            dx, dy = length * np.cos(e_angle), length * np.sin(e_angle)
            ax.plot([-dx/2, dx/2], [-dy/2, dy/2], color='cyan', lw=1.2, alpha=0.8)

        plt.tight_layout()
        plt.show()


generate_button.on_click(on_generate_button_clicked)

col1 = widgets.VBox([zernike_inputs[j] for j in range(4, 11)])
col2 = widgets.VBox([zernike_inputs[j] for j in range(11, 18)])
col3 = widgets.VBox([zernike_inputs[j] for j in range(18, 23)])
display(widgets.HBox([col1, col2, col3]))
display(generate_button)
display(output_area)